In [ ]:
!pip install -q boto3

import os
import json
import random
import hashlib
import re
import boto3

from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
userdata.get('AWS_ACCESS_KEY_ID')
userdata.get('AWS_SECRET_ACCESS_KEY')


In [14]:
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = ("us-east-1")

s3 = boto3.client(
    "s3",
    region_name=os.environ["AWS_DEFAULT_REGION"]
)

print("AWS ready")

AWS ready


In [15]:
BUCKET = "legai-ai-embeddings"

METADATA_PREFIX = "metadata/"

# Fine-tuning sampling
TARGET_SAMPLES = 5000
MAX_PER_METADATA_FILE = 25

MIN_TEXT_LENGTH = 200
MAX_TEXT_LENGTH = 6000

RANDOM_SEED = 42

# Local files
LOCAL_SAMPLE = "/content/sampled_contexts_5000.jsonl"
LOCAL_STATE = "/content/sample_state.json"

# S3 output
S3_SAMPLE_KEY = "finetune/sampled_contexts_5000.jsonl"
S3_STATE_KEY = "finetune/checkpoints/sample_state.json"

# Threading for S3 downloads
DOWNLOAD_WORKERS = 8
FILES_PER_GROUP = 16

In [16]:
def get_metadata_keys():

    paginator = s3.get_paginator("list_objects_v2")

    keys = []

    for page in paginator.paginate(
        Bucket=BUCKET,
        Prefix=METADATA_PREFIX
    ):

        for obj in page.get("Contents", []):

            key = obj["Key"]

            if key.endswith(".jsonl"):
                keys.append(key)

    return keys


metadata_keys = get_metadata_keys()

print("Metadata files found:", len(metadata_keys))

Metadata files found: 7015


In [17]:
rng = random.Random(RANDOM_SEED)

rng.shuffle(metadata_keys)

print(metadata_keys[:10])

['metadata/metadata_4215.jsonl', 'metadata/metadata_5787.jsonl', 'metadata/metadata_2897.jsonl', 'metadata/metadata_2486.jsonl', 'metadata/metadata_4521.jsonl', 'metadata/metadata_40.jsonl', 'metadata/metadata_6932.jsonl', 'metadata/metadata_6360.jsonl', 'metadata/metadata_2696.jsonl', 'metadata/metadata_224.jsonl']


In [18]:
def clean_text(text):

    if text is None:
        return None

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    if len(text) < MIN_TEXT_LENGTH:
        return None

    # Prevent extremely large fine-tuning contexts
    if len(text) > MAX_TEXT_LENGTH:
        text = text[:MAX_TEXT_LENGTH]

    return text

In [19]:
def load_metadata_file(key):

    try:

        response = s3.get_object(
            Bucket=BUCKET,
            Key=key
        )

        raw = (
            response["Body"]
            .read()
            .decode("utf-8")
        )

        records = []

        for line in raw.splitlines():

            try:
                record = json.loads(line)

            except json.JSONDecodeError:
                continue

            text = clean_text(
                record.get("text")
            )

            if not text:
                continue

            record["text"] = text

            records.append(record)

        return key, records

    except Exception as e:

        print(
            "Failed:",
            key,
            e
        )

        return key, []

In [20]:
def s3_exists(key):

    try:

        s3.head_object(
            Bucket=BUCKET,
            Key=key
        )

        return True

    except:
        return False

In [23]:
def load_resume_state():

    processed_keys = set()
    samples = []

    # -------------------------
    # SAMPLE FILE
    # -------------------------

    if s3_exists(S3_SAMPLE_KEY):

        print("Downloading previous sample...")

        s3.download_file(
            BUCKET,
            S3_SAMPLE_KEY,
            LOCAL_SAMPLE
        )

        with open(
            LOCAL_SAMPLE,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                try:
                    samples.append(
                        json.loads(line)
                    )
                except:
                    pass

    # -------------------------
    # STATE FILE
    # -------------------------

    if s3_exists(S3_STATE_KEY):

        print("Downloading checkpoint...")

        s3.download_file(
            BUCKET,
            S3_STATE_KEY,
            LOCAL_STATE
        )

        with open(
            LOCAL_STATE,
            "r"
        ) as f:

            state = json.load(f)

        processed_keys = set(
            state.get(
                "processed_keys",
                []
            )
        )

    print(
        "Recovered samples:",
        len(samples)
    )

    print(
        "Processed metadata files:",
        len(processed_keys)
    )

    return samples, processed_keys

In [21]:
def save_checkpoint(
    samples,
    processed_keys
):

    # =====================================
    # SAVE SAMPLES
    # =====================================

    sample_tmp = (
        LOCAL_SAMPLE + ".tmp"
    )

    with open(
        sample_tmp,
        "w",
        encoding="utf-8"
    ) as f:

        for record in samples:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

    os.replace(
        sample_tmp,
        LOCAL_SAMPLE
    )

    # Upload sample FIRST
    s3.upload_file(
        LOCAL_SAMPLE,
        BUCKET,
        S3_SAMPLE_KEY
    )

    # =====================================
    # SAVE STATE
    # =====================================

    state = {

        "target_samples":
            TARGET_SAMPLES,

        "current_samples":
            len(samples),

        "processed_metadata_files":
            len(processed_keys),

        "processed_keys":
            sorted(processed_keys)
    }

    state_tmp = (
        LOCAL_STATE + ".tmp"
    )

    with open(
        state_tmp,
        "w"
    ) as f:

        json.dump(
            state,
            f,
            indent=2
        )

    os.replace(
        state_tmp,
        LOCAL_STATE
    )

    # State uploaded LAST
    s3.upload_file(
        LOCAL_STATE,
        BUCKET,
        S3_STATE_KEY
    )

    print(
        f"Checkpoint | "
        f"samples={len(samples)} | "
        f"files={len(processed_keys)}"
    )

In [24]:
samples, processed_keys = (
    load_resume_state()
)

Recovered samples: 0
Processed metadata files: 0


In [25]:
seen_hashes = set()
seen_vector_ids = set()


for record in samples:

    text = record["text"]

    text_hash = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    seen_hashes.add(
        text_hash
    )

    vector_id = record.get(
        "vector_id"
    )

    if vector_id is not None:

        seen_vector_ids.add(
            int(vector_id)
        )

In [26]:
def sample_from_records(
    records,
    samples,
    seen_hashes,
    seen_vector_ids
):

    local_rng = random.Random(
        RANDOM_SEED
        + len(samples)
    )

    local_rng.shuffle(
        records
    )

    added = 0

    for record in records:

        if added >= MAX_PER_METADATA_FILE:
            break

        if len(samples) >= TARGET_SAMPLES:
            break

        text = record["text"]

        # -------------------------
        # TEXT DEDUP
        # -------------------------

        text_hash = hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()

        if text_hash in seen_hashes:
            continue

        # -------------------------
        # VECTOR ID DEDUP
        # -------------------------

        vector_id = record.get(
            "vector_id"
        )

        if vector_id is not None:

            vector_id = int(
                vector_id
            )

            if vector_id in seen_vector_ids:
                continue

        # -------------------------
        # SAVE RECORD
        # -------------------------

        output = {

            "vector_id":
                vector_id,

            "batch_id":
                record.get("batch_id"),

            "source":
                record.get("source"),

            "source_file":
                record.get("source_file"),

            "row_index":
                record.get("row_index"),

            "text":
                text
        }

        samples.append(
            output
        )

        seen_hashes.add(
            text_hash
        )

        if vector_id is not None:

            seen_vector_ids.add(
                vector_id
            )

        added += 1

    return added

In [27]:
remaining_keys = [

    key
    for key in metadata_keys
    if key not in processed_keys
]

print(
    "Remaining metadata files:",
    len(remaining_keys)
)

Remaining metadata files: 7015


In [28]:
for start in range(
    0,
    len(remaining_keys),
    FILES_PER_GROUP
):

    if len(samples) >= TARGET_SAMPLES:
        break

    group = remaining_keys[
        start:
        start + FILES_PER_GROUP
    ]

    results = []

    # =====================================
    # PARALLEL S3 DOWNLOAD
    # =====================================

    with ThreadPoolExecutor(
        max_workers=DOWNLOAD_WORKERS
    ) as executor:

        futures = [

            executor.submit(
                load_metadata_file,
                key
            )

            for key in group
        ]

        for future in as_completed(
            futures
        ):

            results.append(
                future.result()
            )

    # =====================================
    # SAMPLE RESULTS
    # =====================================

    for key, records in results:

        if len(samples) >= TARGET_SAMPLES:
            break

        added = sample_from_records(

            records,

            samples,

            seen_hashes,

            seen_vector_ids
        )

        processed_keys.add(
            key
        )

        print(
            f"{key} | "
            f"added={added} | "
            f"total={len(samples)}"
        )

    # =====================================
    # CHECKPOINT
    # =====================================

    save_checkpoint(
        samples,
        processed_keys
    )

metadata/metadata_2897.jsonl | added=25 | total=25
metadata/metadata_2486.jsonl | added=25 | total=50
metadata/metadata_2696.jsonl | added=25 | total=75
metadata/metadata_224.jsonl | added=25 | total=100
metadata/metadata_40.jsonl | added=25 | total=125
metadata/metadata_6360.jsonl | added=25 | total=150
metadata/metadata_5787.jsonl | added=25 | total=175
metadata/metadata_6932.jsonl | added=25 | total=200
metadata/metadata_2236.jsonl | added=0 | total=200
metadata/metadata_6894.jsonl | added=25 | total=225
metadata/metadata_5708.jsonl | added=25 | total=250
metadata/metadata_6219.jsonl | added=25 | total=275
metadata/metadata_4521.jsonl | added=25 | total=300
metadata/metadata_4215.jsonl | added=25 | total=325
metadata/metadata_4786.jsonl | added=25 | total=350
metadata/metadata_4216.jsonl | added=25 | total=375
Checkpoint | samples=375 | files=16
metadata/metadata_145.jsonl | added=24 | total=399
metadata/metadata_405.jsonl | added=22 | total=421
metadata/metadata_3746.jsonl | added=

In [29]:
samples = samples[
    :TARGET_SAMPLES
]

save_checkpoint(
    samples,
    processed_keys
)

print("\n==============================")
print("SAMPLING COMPLETE")
print("==============================")

print(
    "Total samples:",
    len(samples)
)

print(
    "Saved to:"
)

print(
    f"s3://{BUCKET}/"
    f"{S3_SAMPLE_KEY}"
)

Checkpoint | samples=5000 | files=216

SAMPLING COMPLETE
Total samples: 5000
Saved to:
s3://legai-ai-embeddings/finetune/sampled_contexts_5000.jsonl


In [30]:
for i in range(5):

    r = samples[i]

    print(
        "\n",
        "=" * 100
    )

    print(
        "VECTOR ID:",
        r["vector_id"]
    )

    print(
        "SOURCE:",
        r["source"]
    )

    print(
        "SOURCE FILE:",
        r["source_file"]
    )

    print(
        "\nTEXT:\n"
    )

    print(
        r["text"][:1500]
    )


VECTOR ID: 740882
SOURCE: csv2
SOURCE FILE: csv2

TEXT:

Rotor Drive System (1) At least three 10-hour cycles required by this section must be conducted with transmission and gearbox lubricant temperatures, at the location prescribed for measurement, not lower than the maximum operating temperature for which approval is requested;

VECTOR ID: 741480
SOURCE: csv2
SOURCE FILE: csv2

TEXT:

General (c) Each electrical and electronic system that performs such a function whose failure would reduce the capability of the rotorcraft or the ability of the flightcrew to respond to an adverse operating condition must be designed and installed so the system is not adversely affected when the equipment providing these functions is exposed to equipment HIRF test level 3, as described in appendix E to this part.

VECTOR ID: 740847
SOURCE: csv2
SOURCE FILE: csv2

TEXT:

Rotor Drive System (1) Except as prescribed in paragraphs (b)(2) and (b)(3) of this section, the takeoff torque run must consist of 